## Example 1: asyncssh functionality test 1

In [ ]:
import asyncssh

# Example 1: Basic asyncssh SSH client functionality
print("Example 1: Basic asyncssh SSH client functionality")

try:
    print("asyncssh module imported successfully")
    print("Async SSH client and server library")
    
    # Check available components
    attrs = [attr for attr in dir(asyncssh) if not attr.startswith('_')]
    print(f"Available components: {len(attrs)}")
    print(f"Key components: {attrs[:10] if len(attrs) > 10 else attrs}")
    
    # Show connection-related functionality
    connection_attrs = [attr for attr in attrs if 'connect' in attr.lower()]
    print(f"Connection-related: {connection_attrs}")
    
    # Show key/auth related functionality  
    auth_attrs = [attr for attr in attrs if any(word in attr.lower() for word in ['key', 'auth', 'cert'])]
    print(f"Authentication-related: {auth_attrs}")
    
    # Show server-related functionality
    server_attrs = [attr for attr in attrs if 'server' in attr.lower() or 'listen' in attr.lower()]
    print(f"Server-related: {server_attrs}")
    
    # Check for key functions
    if hasattr(asyncssh, 'connect'):
        print("✓ connect function available for SSH connections")
    
    if hasattr(asyncssh, 'listen'):
        print("✓ listen function available for SSH server")
    
    if hasattr(asyncssh, 'generate_private_key'):
        print("✓ Key generation functions available")
    
    # Show version if available
    if hasattr(asyncssh, '__version__'):
        print(f"asyncssh version: {asyncssh.__version__}")
    
    # Test key generation capabilities
    if hasattr(asyncssh, 'generate_private_key'):
        try:
            # Test key generation patterns (without actual generation)
            key_types = ['ssh-rsa', 'ssh-ed25519', 'ecdsa-sha2-nistp256']
            print(f"Supported key types: {key_types}")
        except Exception as e:
            print(f"Key generation test: {type(e).__name__}")
            raise
    
    print("asyncssh example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncssh example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: asyncssh functionality test 2

In [ ]:
import asyncssh

# Example 2: asyncssh connection configuration and key management
print("Example 2: asyncssh connection configuration and key management")

try:
    # Test asyncssh connection configuration (without requiring actual SSH server)
    async def test_connection_parameters():
        # Show connection parameter structure that asyncssh accepts
        connection_params = {
            "host": "example.com",
            "port": 22,
            "username": "user",
            "password": "secret",  # For password auth
            "client_keys": ["~/.ssh/id_rsa", "~/.ssh/id_ed25519"],
            "known_hosts": "~/.ssh/known_hosts",
            "connect_timeout": 30,
            "keepalive_interval": 30,
            "compression_algs": ["zlib@openssh.com", "zlib", "none"],
            "encryption_algs": ["aes256-ctr", "aes128-ctr", "chacha20-poly1305@openssh.com"]
        }
        
        print("✓ Connection parameters structure:")
        for key, value in connection_params.items():
            print(f"  {key}: {value}")
        
        # Test connection function signature
        try:
            # This would normally connect to SSH server, but we'll catch the connection error
            # to demonstrate the asyncssh.connect function exists and accepts parameters
            async with asyncssh.connect(
                host="nonexistent_host",
                username="test",
                password="test",
                connect_timeout=1,  # Short timeout to fail quickly
                known_hosts=None  # Skip host key verification for test
            ) as conn:
                pass
        except (OSError, asyncssh.Error) as e:
            # Expected to fail since we're not connecting to a real SSH server
            print(f"✓ asyncssh.connect() function tested (expected connection failure: {type(e).__name__})")
        else:
            raise AssertionError("asyncssh.connect() to 'nonexistent_host' unexpectedly succeeded")
        
        return True
    
    # Test asyncssh key generation and management
    async def test_key_generation():
        print("\n✓ asyncssh key generation and management:")
        
        # Test key generation functions
        key_algorithms = [
            "ssh-rsa",
            "ssh-dss", 
            "ecdsa-sha2-nistp256",
            "ecdsa-sha2-nistp384",
            "ecdsa-sha2-nistp521",
            "ssh-ed25519"
        ]
        
        print("  Supported key algorithms:")
        for alg in key_algorithms:
            print(f"    • {alg}")
        
        # Generate a test key to verify the function works
        test_key = asyncssh.generate_private_key('ssh-ed25519')
        assert test_key.get_algorithm() == 'ssh-ed25519', test_key.get_algorithm()
        print(f"  ✓ Key generation successful: {type(test_key).__name__}")
        
        # Test key export formats
        public_key_openssh = test_key.export_public_key('openssh')
        assert public_key_openssh.startswith(b'ssh-ed25519 '), public_key_openssh[:20]
        print(f"  ✓ Public key export (OpenSSH format): {len(public_key_openssh)} chars")
        
        # The exported public key imports back to the same key
        reimported = asyncssh.import_public_key(public_key_openssh)
        assert reimported.export_public_key('openssh') == public_key_openssh
        
        # Test fingerprint generation
        fingerprint = test_key.get_fingerprint()
        assert fingerprint.startswith('SHA256:'), fingerprint
        assert reimported.get_fingerprint() == fingerprint
        print(f"  ✓ Key fingerprint: {fingerprint}")
        
        return True
    
    # Test asyncssh authentication methods
    async def test_authentication_methods():
        print("\n✓ asyncssh authentication methods:")
        
        auth_methods = [
            "password: Username and password authentication",
            "publickey: SSH key-based authentication", 
            "keyboard_interactive: Interactive prompts",
            "gssapi: Kerberos/GSSAPI authentication",
            "host_based: Host-based authentication",
            "none: No authentication (rare)"
        ]
        
        for method in auth_methods:
            print(f"  • {method}")
        
        # Test authentication configuration patterns
        auth_configs = [
            "client_keys=['~/.ssh/id_rsa'] - Specify key files",
            "password='secret' - Password authentication",
            "passphrase='keypass' - Key passphrase",
            "agent_path='/path/to/agent' - SSH agent socket",
            "known_hosts='~/.ssh/known_hosts' - Host verification"
        ]
        
        print("\n  Authentication configuration patterns:")
        for config in auth_configs:
            print(f"    {config}")
        
        return True
    
    # Test asyncssh connection patterns
    async def test_connection_patterns():
        print("\n✓ asyncssh connection patterns:")
        
        patterns = [
            "async with asyncssh.connect(host) as conn - Context manager",
            "conn = await asyncssh.connect(host) - Direct connection",
            "result = await conn.run(command) - Command execution",
            "async with conn.start_sftp_client() as sftp - SFTP client",
            "listener = await asyncssh.listen(...) - SSH server",
            "await conn.forward_local_port(...) - Port forwarding"
        ]
        
        for pattern in patterns:
            print(f"  • {pattern}")
        
        return True
    
    # Run all tests
    await test_connection_parameters()
    await test_key_generation()
    await test_authentication_methods()
    await test_connection_patterns()
    
    print("\nasyncssh connection configuration and key management demonstrated")
    print("asyncssh example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncssh example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: asyncssh functionality test 3

In [ ]:
import asyncssh
import socket

# A port guaranteed closed on this host: bind an ephemeral port, read its number, release it.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as _probe:
    _probe.bind(("127.0.0.1", 0))
    closed_port = _probe.getsockname()[1]

# Example 3: asyncssh SFTP operations and file transfer
print("Example 3: asyncssh SFTP operations and file transfer")

try:
    # Test asyncssh SFTP client functionality
    async def test_sftp_functionality():
        print("✓ Testing asyncssh SFTP functionality:")
        
        # Test connection attempt to verify SFTP interface
        try:
            async with asyncssh.connect(
                host="127.0.0.1",
                port=closed_port,  # nothing listens here
                username="test",
                connect_timeout=1,
                known_hosts=None
            ) as conn:
                # If successful, test SFTP client
                async with conn.start_sftp_client() as sftp:
                    files = await sftp.listdir('.')
                    print(f"  SFTP directory listing: {files}")
            
        except ConnectionRefusedError as e:
            print(f"  ✓ Connection attempt for SFTP test: {type(e).__name__}")
        else:
            raise AssertionError(f"expected ConnectionRefusedError from closed port {closed_port}")
        
        print("  SFTP operation patterns:")
        sftp_patterns = [
            "async with conn.start_sftp_client() as sftp:",
            "    files = await sftp.listdir(path)",
            "    await sftp.get(remote_path, local_path)",
            "    await sftp.put(local_path, remote_path)",
            "    await sftp.mkdir(path)",
            "    await sftp.rmdir(path)",
            "    await sftp.remove(path)",
            "    await sftp.rename(old_path, new_path)"
        ]
        
        for pattern in sftp_patterns:
            print(f"    {pattern}")
        
        return True
    
    # Test asyncssh SCP functionality
    async def test_scp_functionality():
        print("\n✓ asyncssh SCP operations:")
        
        scp_patterns = [
            "SCP upload: await conn.scp(local_file, remote_path)",
            "SCP download: await conn.scp(remote_path, local_file)",
            "Recursive copy: await conn.scp(local_dir, remote_dir, recurse=True)",
            "Progress callback: await conn.scp(..., progress_handler=callback)",
            "Preserve attributes: await conn.scp(..., preserve=True)"
        ]
        
        for pattern in scp_patterns:
            print(f"  • {pattern}")
        
        print("\n  SCP vs SFTP comparison:")
        comparison = [
            "SCP: Simple, fast, protocol-based transfers",
            "SFTP: Full filesystem operations, resumable transfers",
            "SCP: Better for scripted bulk transfers", 
            "SFTP: Better for interactive file management"
        ]
        
        for comp in comparison:
            print(f"    {comp}")
        
        return True
    
    # Test asyncssh file transfer features
    async def test_file_transfer_features():
        print("\n✓ asyncssh file transfer features:")
        
        transfer_features = [
            "Recursive directory transfers",
            "Symbolic link handling",
            "File permission preservation",
            "Progress callback support",
            "Resume interrupted transfers",
            "Parallel file transfers",
            "Compression during transfer",
            "Custom transfer protocols"
        ]
        
        for feature in transfer_features:
            print(f"  • {feature}")
        
        print("\n  Transfer configuration options:")
        config_options = [
            "preserve=True: Maintain file timestamps and permissions",
            "recurse=True: Recursive directory operations",
            "follow_symlinks=False: Handle symbolic links",
            "block_size=16384: Transfer block size",
            "max_requests=128: Concurrent request limit",
            "progress_handler=func: Progress callback function"
        ]
        
        for option in config_options:
            print(f"    {option}")
        
        return True
    
    # Test asyncssh error handling for file operations
    async def test_file_error_handling():
        print("\n✓ asyncssh file operation error handling:")
        
        # Show SFTP-specific exception types
        # Import SFTP exceptions to verify they exist
        from asyncssh import SFTPError, SFTPFailure, SFTPNoSuchFile
        print("  ✓ SFTP exception classes available:")
        print(f"    SFTPError: {SFTPError}")
        print(f"    SFTPFailure: {SFTPFailure}")  
        print(f"    SFTPNoSuchFile: {SFTPNoSuchFile}")
        assert issubclass(SFTPFailure, SFTPError) and issubclass(SFTPNoSuchFile, SFTPError)
        
        error_handling_patterns = [
            "try:",
            "    await sftp.get('remote_file', 'local_file')",
            "except SFTPNoSuchFile:",
            "    print('Remote file not found')",
            "except SFTPFailure as e:",
            "    print(f'Transfer failed: {e}')",
            "except OSError as e:",
            "    print(f'Local filesystem error: {e}')"
        ]
        
        print("\n  Error handling patterns:")
        for pattern in error_handling_patterns:
            print(f"    {pattern}")
        
        return True
    
    # Test asyncssh filesystem operations
    async def test_filesystem_operations():
        print("\n✓ asyncssh filesystem operations:")
        
        filesystem_ops = [
            "Directory operations:",
            "  await sftp.listdir(path) - List directory contents",
            "  await sftp.mkdir(path) - Create directory",
            "  await sftp.rmdir(path) - Remove directory",
            "",
            "File operations:",
            "  await sftp.stat(path) - Get file attributes",
            "  await sftp.remove(path) - Delete file",
            "  await sftp.rename(old, new) - Rename/move file",
            "",
            "Advanced operations:",
            "  await sftp.symlink(target, link) - Create symbolic link",
            "  await sftp.readlink(path) - Read symbolic link target",
            "  await sftp.realpath(path) - Resolve absolute path"
        ]
        
        for op in filesystem_ops:
            if op:  # Skip empty lines for cleaner output
                print(f"    {op}")
        
        return True
    
    # Run all SFTP and file transfer tests
    await test_sftp_functionality()
    await test_scp_functionality()
    await test_file_transfer_features()
    await test_file_error_handling()
    await test_filesystem_operations()
    
    print("\nasyncssh SFTP operations and file transfer demonstrated")
    print("asyncssh example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncssh example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: asyncssh functionality test 4

In [ ]:
import asyncssh
import socket

# A port guaranteed closed on this host: bind an ephemeral port, read its number, release it.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as _probe:
    _probe.bind(("127.0.0.1", 0))
    closed_port = _probe.getsockname()[1]

# Example 4: asyncssh server functionality and port forwarding
print("Example 4: asyncssh server functionality and port forwarding")

try:
    # Test asyncssh SSH server functionality
    async def test_ssh_server_functionality():
        print("✓ Testing asyncssh SSH server functionality:")
        
        # Test server creation (will fail without proper setup, but tests the function exists)
        try:
            # Test server listener creation
            server = await asyncssh.listen(
                host="127.0.0.1",
                port=0,  # any free port; loading the missing host key fails before anything binds
                server_host_keys=["nonexistent_key"],  # Will cause expected failure
                server_factory=None
            )
            
        except FileNotFoundError as e:
            # The host key is loaded from disk before anything binds, so the missing file is what fails.
            assert str(e.filename).endswith("nonexistent_key"), e
            print(f"  ✓ asyncssh.listen() function exists and handles errors: {type(e).__name__}")
        else:
            # Unexpected success: release the listener, then fail the example
            server.close()
            await server.wait_closed()
            raise AssertionError("expected FileNotFoundError for a missing server host key file")
        
        print("  SSH server configuration options:")
        server_options = [
            "server_host_keys: List of host key files",
            "authorized_client_keys: Client authentication keys",
            "password_auth: Enable password authentication",
            "public_key_auth: Enable public key authentication", 
            "kbdint_auth: Enable keyboard-interactive authentication",
            "gss_auth: Enable GSSAPI authentication",
            "compression_algs: Supported compression algorithms",
            "encryption_algs: Supported encryption algorithms"
        ]
        
        for option in server_options:
            print(f"    • {option}")
        
        return True
    
    # Test asyncssh port forwarding functionality
    async def test_port_forwarding():
        print("\n✓ asyncssh port forwarding functionality:")
        
        # Test connection for port forwarding interface
        try:
            async with asyncssh.connect(
                host="127.0.0.1",
                port=closed_port,  # nothing listens here
                username="test",
                connect_timeout=1,
                known_hosts=None
            ) as conn:
                # If successful, test port forwarding
                listener = await conn.forward_local_port('', 8080, 'target.com', 80)
                listener.close()
                
        except ConnectionRefusedError as e:
            print(f"  ✓ Connection attempt for port forwarding test: {type(e).__name__}")
        else:
            raise AssertionError(f"expected ConnectionRefusedError from closed port {closed_port}")
        
        print("  Port forwarding types:")
        forwarding_types = [
            "Local forwarding: await conn.forward_local_port(bind_host, bind_port, dest_host, dest_port)",
            "Remote forwarding: await conn.forward_remote_port(bind_host, bind_port, dest_host, dest_port)",
            "Dynamic forwarding: await conn.forward_socks(bind_host, bind_port)",
            "Unix socket forwarding: await conn.forward_unix_domain_socket(path, dest_host, dest_port)"
        ]
        
        for fwd_type in forwarding_types:
            print(f"    • {fwd_type}")
        
        return True
    
    # Test asyncssh tunneling patterns
    async def test_tunneling_patterns():
        print("\n✓ asyncssh tunneling patterns:")
        
        tunneling_scenarios = [
            "Database access through bastion host",
            "Web application development tunnels",
            "Remote desktop access via SSH",
            "API endpoint secure proxying",
            "Internal service exposure",
            "SOCKS proxy for secure browsing"
        ]
        
        print("  Common tunneling scenarios:")
        for scenario in tunneling_scenarios:
            print(f"    • {scenario}")
        
        print("\n  Tunneling configuration patterns:")
        tunnel_configs = [
            "# Local port forwarding (access remote service locally)",
            "listener = await conn.forward_local_port('localhost', 5432, 'db.internal', 5432)",
            "",
            "# Remote port forwarding (expose local service remotely)", 
            "listener = await conn.forward_remote_port('0.0.0.0', 8080, 'localhost', 3000)",
            "",
            "# Dynamic SOCKS proxy",
            "listener = await conn.forward_socks('localhost', 1080)",
            "",
            "# Unix domain socket forwarding",
            "listener = await conn.forward_unix_domain_socket('/tmp/tunnel.sock', 'service.internal', 80)"
        ]
        
        for config in tunnel_configs:
            if config:  # Skip empty lines for cleaner output
                print(f"    {config}")
        
        return True
    
    # Test asyncssh connection multiplexing
    async def test_connection_multiplexing():
        print("\n✓ asyncssh connection multiplexing:")
        
        multiplexing_features = [
            "Multiple channels over single connection",
            "Concurrent command execution",
            "Simultaneous SFTP and shell sessions",
            "Port forwarding with active sessions",
            "Shared connection for multiple operations",
            "Connection pooling and reuse"
        ]
        
        for feature in multiplexing_features:
            print(f"  • {feature}")
        
        print("\n  Multiplexing usage patterns:")
        mux_patterns = [
            "async with asyncssh.connect(host) as conn:",
            "    # Multiple concurrent operations",
            "    task1 = conn.run('command1')",
            "    task2 = conn.run('command2')",
            "    async with conn.start_sftp_client() as sftp:",
            "        # SFTP operations concurrent with commands",
            "        files = await sftp.listdir('.')",
            "    # Wait for commands to complete",
            "    result1, result2 = await asyncio.gather(task1, task2)"
        ]
        
        for pattern in mux_patterns:
            print(f"    {pattern}")
        
        return True
    
    # Test asyncssh advanced server features
    async def test_advanced_server_features():
        print("\n✓ asyncssh advanced server features:")
        
        advanced_features = [
            "Custom authentication handlers",
            "Session and channel management",
            "Subsystem support (like SFTP)",
            "X11 forwarding capabilities",
            "Agent forwarding support",
            "Custom command handlers",
            "Connection event callbacks",
            "Security policy enforcement"
        ]
        
        for feature in advanced_features:
            print(f"  • {feature}")
        
        print("\n  Server customization patterns:")
        server_patterns = [
            "class CustomSSHServer(asyncssh.SSHServer):",
            "    def connection_made(self, conn):",
            "        print(f'Connection from {conn.get_extra_info(\"peername\")}')",
            "",
            "    def begin_auth(self, username):",
            "        return True  # Allow authentication",
            "",
            "    def password_auth_supported(self):",
            "        return True",
            "",
            "    def validate_password(self, username, password):",
            "        return password == 'secret'"
        ]
        
        for pattern in server_patterns:
            if pattern:  # Skip empty lines for cleaner output
                print(f"    {pattern}")
        
        return True
    
    # Run all server and port forwarding tests
    await test_ssh_server_functionality()
    await test_port_forwarding()
    await test_tunneling_patterns()
    await test_connection_multiplexing()
    await test_advanced_server_features()
    
    print("\nasyncssh server functionality and port forwarding demonstrated")
    print("asyncssh example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncssh example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: asyncssh functionality test 5

In [ ]:
import asyncssh

# Example 5: asyncssh package capabilities and SSH protocol features
print("Example 5: asyncssh package capabilities and SSH protocol features")

try:
    print("asyncssh provides comprehensive async SSH client/server functionality")
    
    # Package analysis
    attrs = [attr for attr in dir(asyncssh) if not attr.startswith('_')]
    print(f"Available components: {len(attrs)}")
    print(f"Key components: {attrs[:12] if len(attrs) > 12 else attrs}")
    
    # Show version information
    if hasattr(asyncssh, '__version__'):
        print(f"asyncssh version: {asyncssh.__version__}")
    
    # Show key capabilities
    print("\nKey asyncssh capabilities:")
    capabilities = [
        "🔐 Full SSH protocol implementation (SSH2)",
        "🚀 High-performance async client and server",
        "🔑 Multiple authentication methods",
        "📁 SFTP and SCP file transfer support",
        "🌉 Port forwarding and tunneling",
        "🖥️ Interactive shell and command execution",
        "📜 X.509 certificate authentication",
        "🔧 Extensive configuration options"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # SSH protocol features supported
    print(f"\nSSH protocol features supported:")
    ssh_features = [
        "SSH2 protocol with all standard algorithms",
        "Password, public key, and keyboard-interactive auth",
        "RSA, DSA, ECDSA, and Ed25519 keys",
        "AES, ChaCha20, and 3DES encryption",
        "HMAC-SHA1/SHA2 message authentication",
        "Diffie-Hellman key exchange variants",
        "Compression with zlib and zlib@openssh.com",
        "SSH certificates for user and host authentication"
    ]
    
    for feature in ssh_features:
        print(f"• {feature}")
    
    # File transfer capabilities
    print(f"\nFile transfer capabilities:")
    file_features = [
        "SFTP v3-v6 protocol support",
        "SCP file transfer protocol",
        "Recursive directory operations",
        "Resume interrupted transfers",
        "Parallel file transfers",
        "Symbolic link handling",
        "File permission preservation",
        "Progress callback support"
    ]
    
    for feature in file_features:
        print(f"• {feature}")
    
    # Usage patterns
    print(f"\nCommon usage patterns:")
    patterns = [
        "Remote server administration and monitoring",
        "Automated deployment and configuration management",
        "Secure file synchronization and backup",
        "Database tunneling and secure access",
        "Development environment management",
        "CI/CD pipeline remote execution",
        "Jump host and bastion server access",
        "IoT device management and updates"
    ]
    
    for pattern in patterns:
        print(f"• {pattern}")
    
    # Advanced features
    print(f"\nAdvanced features:")
    advanced = [
        "SSH agent forwarding",
        "X11 forwarding for GUI applications",
        "TCP and Unix domain socket forwarding",
        "SOCKS proxy functionality",
        "OpenSSH compatibility",
        "Known hosts file management", 
        "Connection multiplexing",
        "Graceful connection handling"
    ]
    
    for feature in advanced:
        print(f"• {feature}")
    
    # Integration examples
    print(f"\nFramework integration:")
    integrations = [
        "Ansible for configuration management",
        "Fabric for deployment automation", 
        "Paramiko replacement with better performance",
        "Docker container management",
        "Kubernetes cluster administration",
        "Cloud infrastructure provisioning",
        "DevOps toolchain integration"
    ]
    
    for integration in integrations:
        print(f"• {integration}")
    
    # Performance characteristics
    print(f"\nPerformance characteristics:")
    performance = [
        "Pure Python implementation with C extensions",
        "Optimized for high-throughput applications",
        "Efficient memory usage for large transfers",
        "Concurrent connection handling",
        "Non-blocking I/O operations",
        "Connection pooling support"
    ]
    
    for char in performance:
        print(f"• {char}")
    
    print(f"\nasyncssh is a comprehensive solution for secure remote operations")
    print("asyncssh example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncssh example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")